In [3]:
import wrds
import pandas as pd
import os
import time     
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import time

In [5]:
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [9]:
query = """
SELECT DISTINCT
    c.cik,
    c.conm,
    l.lpermno AS permno,
    m.start,
    m.ending
FROM crsp.msp500list m

JOIN crsp.ccmxpf_linktable l
    ON m.permno = l.lpermno
    AND l.linktype IN ('LU', 'LC')
    AND l.linkprim IN ('P', 'C')

JOIN comp.company c
    ON l.gvkey = c.gvkey

WHERE c.cik IS NOT NULL
    AND m.start <= '2025-12-31'
    AND (m.ending IS NULL OR m.ending >= '2000-01-01')
"""

df = db.raw_sql(query)

bad_keywords = [
    "FUND", "ETF", "TRUST", "PORTFOLIO",
    "SERIES", "INVESTMENT", "SICAV"
]

df = df[~df['conm'].str.upper().str.contains('|'.join(bad_keywords), na=False)]

ciks = df['cik'].dropna().astype(str).str.zfill(10).unique()

cik_df = pd.DataFrame({
    "cik": ciks
})

cik_df.to_csv("sp500_historical_ciks.csv", index=False)

print(f"Saved {len(ciks)} CIKs")

Saved 1046 CIKs


In [ ]:
BASE_DIR = Path(__file__).resolve().parent.parent
data_dir = BASE_DIR / "data" / "RensTest"
data_dir.mkdir(parents=True, exist_ok=True)

set_identity("gerritsenREN2@gmail.com")

tickers = df['cik'].astype(str).tolist()
start_yr = 2001


MAX_WORKERS = 5  # safe

def process_ticker(ticker):
    try:
        company = Company(ticker)
        filings = company.get_filings(form="10-K")

        for filing in filings:
            year = filing.filing_date.year
            if year < start_yr:
                continue

            strat_path = data_dir / f"{ticker}_{year}_strategy.txt"
            risk_path = data_dir / f"{ticker}_{year}_risk.txt"
            mgmt_path = data_dir / f"{ticker}_{year}_MD&A.txt"

            if strat_path.exists() and risk_path.exists():
                continue

            tenk = filing.obj()

            strategy_text = tenk.get('Item 1')
            risk_text = tenk.get('Item 1A')
            mgmt_text = tenk.get('Item 7')

            if strategy_text:
                strat_path.write_text(strategy_text, encoding="utf-8")
            if risk_text:
                risk_path.write_text(risk_text, encoding="utf-8")
            if mgmt_text:
                mgmt_path.write_text(mgmt_text, encoding="utf-8")

            time.sleep(0.2)  # instead of 1 sec

    except Exception as e:
        print(f"Error with {ticker}: {e}")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    executor.map(process_ticker, tickers)

NameError: name 'Path' is not defined